# RAGAS Evaluation for QBrain

This notebook adds a secondary evaluation layer using RAGAS on top of the existing benchmark outputs.

It reads `benchmark_full_results.csv`, computes RAGAS metrics, and saves paper-ready tables.


In [1]:
from pathlib import Path
import os
import pandas as pd
from dotenv import load_dotenv

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
elif ROOT.name != "rag_lab":
    ROOT = Path("d:/Qbrainpython/QBrain/rag_lab")

# Load .env so OPENAI_API_KEY is available inside notebook kernels.
load_dotenv(ROOT / ".env")

candidates = [
    ROOT / "data" / "outputs" / "evaluation" / "rag_benchmark" / "benchmark_full_results.csv",
    ROOT / "results" / "paper_output_sample" / "benchmark_full_results.csv",
    ROOT / "results" / "paper_output_sample" / "jdeco_answer_correctness.csv",
    ROOT / "data" / "outputs" / "evaluation_results.csv",
]

BENCHMARK_CSV = next((p for p in candidates if p.exists()), None)
if BENCHMARK_CSV is None:
    raise FileNotFoundError(
        "No evaluation CSV found. Expected one of: " + ", ".join(str(p) for p in candidates)
    )

OUT_TABLES = ROOT / "results" / "tables"
OUT_TABLES.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("BENCHMARK_CSV:", BENCHMARK_CSV)
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))
if not os.getenv("OPENAI_API_KEY"):
    print("WARNING: OPENAI_API_KEY is missing; set it in rag_lab/.env before running RAGAS cells.")


ROOT: d:\Qbrainpython\QBrain\rag_lab
BENCHMARK_CSV: d:\Qbrainpython\QBrain\rag_lab\data\outputs\evaluation\rag_benchmark\benchmark_full_results.csv
OPENAI_API_KEY set: True


## 1) Install dependencies (run once)

If `ragas` is not installed, uncomment and run the next line.


In [2]:
# %pip install ragas datasets langchain-openai -U


## 2) Prepare dataset from benchmark output


In [3]:
from datasets import Dataset

df = pd.read_csv(BENCHMARK_CSV)

# Normalize expected column names from different pipelines
if "generated_answer" not in df.columns and "answer" in df.columns:
    df["generated_answer"] = df["answer"]
if "expected_answer" not in df.columns and "ground_truth" in df.columns:
    df["expected_answer"] = df["ground_truth"]

required_cols = ["question", "generated_answer", "expected_answer"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(
        f"Missing expected columns in selected CSV: {missing}. "
        "Use a file containing question/generated_answer/expected_answer."
    )

# Build contexts from retrieved text/files when available.
def _contexts_from_row(row):
    # Preferred: real retrieved chunk texts saved by run_rag_benchmark.py
    raw_ctx = row.get("retrieved_contexts_json", None)
    if isinstance(raw_ctx, str) and raw_ctx.strip() and raw_ctx.strip() != "nan":
        import json
        try:
            arr = json.loads(raw_ctx)
            if isinstance(arr, list):
                return [str(x) for x in arr if str(x).strip()]
        except Exception:
            pass

    # Fallback: retrieved files list (weaker for faithfulness)
    val = str(row.get("retrieved_files_topk", ""))
    if val and val != "nan":
        return [x.strip() for x in val.split(";") if x.strip()]

    # Last fallback: source file name if present
    src = str(row.get("srs_file", ""))
    return [src] if src and src != "nan" else []

ragas_df = pd.DataFrame(
    {
        "question": df["question"].astype(str),
        "answer": df["generated_answer"].astype(str),
        "ground_truth": df["expected_answer"].astype(str),
        "contexts": df.apply(_contexts_from_row, axis=1),
        "srs_file": df["srs_file"].astype(str) if "srs_file" in df.columns else "unknown",
        "question_id": df["question_id"].astype(str) if "question_id" in df.columns else df.index.astype(str),
    }
)

ragas_dataset = Dataset.from_pandas(ragas_df[["question", "answer", "ground_truth", "contexts"]], preserve_index=False)
display(ragas_df.head(3))
print("Rows:", len(ragas_df))


,question,answer,ground_truth,contexts,srs_file,question_id
0,What document is specified at the beginning of...,The document specified at the beginning of the...,The document is the ERTMS/ETCS Functional Requ...,[12.10.2005 \nOctober 2005 \n4.40 Oct \nUpdate...,2007 - ertms.pdf,q_ertms_01
1,What filing number is shown for this ERTMS/ETC...,The filing number shown for the ERTMS/ETCS FRS...,The filing number is ERA/ERTMS/003204.,[ERTMS/ETCS FRS v 5.00 \n \n \n \n \n \n \n \n...,2007 - ertms.pdf,q_ertms_02
2,What does the amendment history say about vers...,The amendment history indicates that version 5...,Version 5.00 is marked as the official release...,[12.10.2005 \nOctober 2005 \n4.40 Oct \nUpdate...,2007 - ertms.pdf,q_ertms_03


Rows: 45


## 3) Run RAGAS metrics

Default metrics below are commonly reported in papers:
- `faithfulness`
- `answer_relevancy`
- `answer_correctness`


In [4]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, answer_correctness
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import os
import warnings

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError("OPENAI_API_KEY is not set. Add it to rag_lab/.env then rerun Cell 1.")

# For ragas==0.4.x these imports are already Metric objects accepted by evaluate().
warnings.filterwarnings("ignore", category=DeprecationWarning)
metrics = [faithfulness, answer_relevancy, answer_correctness]

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
emb = OpenAIEmbeddings(model="text-embedding-3-small")

result = evaluate(
    ragas_dataset,
    metrics=metrics,
    llm=llm,
    embeddings=emb,
    raise_exceptions=False,
)

result_df = result.to_pandas()
display(result_df.head())
print("Columns:", list(result_df.columns))


C:\Users\ahmad\AppData\Local\Temp\ipykernel_12832\1710957084.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, answer_correctness
C:\Users\ahmad\AppData\Local\Temp\ipykernel_12832\1710957084.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, answer_correctness
C:\Users\ahmad\AppData\Local\Temp\ipykernel_12832\1710957084.py:2: DeprecationWarning: Importing answer_correctness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections im

Evaluating:   0%|          | 0/135 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[2]: TimeoutError()
Exception raised in Job[11]: TimeoutError()
Exception raised in Job[8]: TimeoutError()
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3.

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,answer_correctness
0,What document is specified at the beginning of...,[12.10.2005 \nOctober 2005 \n4.40 Oct \nUpdate...,The document specified at the beginning of the...,The document is the ERTMS/ETCS Functional Requ...,1.000,0.877680,NaN
1,What filing number is shown for this ERTMS/ETC...,[ERTMS/ETCS FRS v 5.00 \n \n \n \n \n \n \n \n...,The filing number shown for the ERTMS/ETCS FRS...,The filing number is ERA/ERTMS/003204.,1.000,0.971470,0.967751
2,What does the amendment history say about vers...,[12.10.2005 \nOctober 2005 \n4.40 Oct \nUpdate...,The amendment history indicates that version 5...,Version 5.00 is marked as the official release...,0.875,0.597858,NaN
3,"According to the table of contents, which sect...",[ERTMS/ETCS FRS v 5.00 \n \n \n \n \nTABLE OF ...,"According to the table of contents, the sectio...","Section 2, ETCS Objectives, comes immediately ...",1.000,0.900363,NaN
4,Which subsection of General requirements addre...,[ERTMS/ETCS FRS v 5.00 \n \n11\n \n3.3 \nInten...,The subsection of General requirements that ad...,Subsection 3.7 addresses operation with existi...,1.000,0.992323,0.966017


Columns: ['user_input', 'retrieved_contexts', 'response', 'reference', 'faithfulness', 'answer_relevancy', 'answer_correctness']


## 4) Export paper-ready tables


In [5]:
if "result_df" not in globals():
    raise RuntimeError("result_df is missing. Run the previous RAGAS evaluation cell first.")

full = pd.concat([ragas_df.reset_index(drop=True), result_df.reset_index(drop=True)], axis=1)

metric_cols = [c for c in ["faithfulness", "answer_relevancy", "answer_correctness"] if c in full.columns]
if not metric_cols:
    raise ValueError("RAGAS metric columns not found in result dataframe.")

overall = full[metric_cols].mean().reset_index()
overall.columns = ["metric", "value"]

per_srs = full.groupby("srs_file", as_index=False)[metric_cols].mean()

full.to_csv(OUT_TABLES / "ragas_by_question.csv", index=False)
overall.to_csv(OUT_TABLES / "ragas_overall_summary.csv", index=False)
per_srs.to_csv(OUT_TABLES / "ragas_summary_by_srs.csv", index=False)

(OUT_TABLES / "ragas_overall_summary.tex").write_text(
    overall.to_latex(index=False, float_format=lambda x: f"{x:.4f}"),
    encoding="utf-8",
)
(OUT_TABLES / "ragas_summary_by_srs.tex").write_text(
    per_srs.to_latex(index=False, float_format=lambda x: f"{x:.4f}"),
    encoding="utf-8",
)

display(overall)
display(per_srs)
print("Saved RAGAS outputs to:", OUT_TABLES)


,metric,value
0,faithfulness,0.955808
1,answer_relevancy,0.817359
2,answer_correctness,0.696713


,srs_file,faithfulness,answer_relevancy,answer_correctness
0,2007 - ertms.pdf,0.954167,0.717961,0.868103
1,2008 - keepass.pdf,0.975000,0.816503,0.693159
2,2009 - inventory 2.0.pdf,0.924242,0.846739,0.555038
3,2010 - gparted.pdf,0.975000,0.868330,0.733674
4,JDECo_SRS.docx[1].pdf,1.000000,0.930834,NaN


Saved RAGAS outputs to: d:\Qbrainpython\QBrain\rag_lab\results\tables




- "We complemented similarity-threshold evaluation with RAGAS metrics (faithfulness, answer relevancy, and answer correctness)."
- "This provides an additional quality perspective beyond strict semantic-threshold accuracy and helps distinguish factual grounding from stylistic variance."


## Rebuilt Stable Pipeline (Use this section only)

This is a clean, version-compatible path for `ragas==0.4.x`.
Run cells below in order and ignore previous RAGAS cells.

In [6]:
from pathlib import Path
import os
import json
import pandas as pd
from dotenv import load_dotenv
from datasets import Dataset

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
elif ROOT.name != "rag_lab":
    ROOT = Path("d:/Qbrainpython/QBrain/rag_lab")

load_dotenv(ROOT / ".env")

candidates = [
    ROOT / "data" / "outputs" / "evaluation" / "rag_benchmark" / "benchmark_full_results.csv",
    ROOT / "results" / "paper_output_sample" / "benchmark_full_results.csv",
    ROOT / "data" / "outputs" / "evaluation_results.csv",
]

BENCHMARK_CSV = next((p for p in candidates if p.exists()), None)
if BENCHMARK_CSV is None:
    raise FileNotFoundError("benchmark_full_results.csv not found. Run benchmark first.")

OUT_TABLES = ROOT / "results" / "tables"
OUT_TABLES.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("BENCHMARK_CSV:", BENCHMARK_CSV)
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))
if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError("OPENAI_API_KEY is missing. Add it to rag_lab/.env and rerun this cell.")

# Load/normalize input dataframe
df = pd.read_csv(BENCHMARK_CSV)
if "generated_answer" not in df.columns and "answer" in df.columns:
    df["generated_answer"] = df["answer"]
if "expected_answer" not in df.columns and "ground_truth" in df.columns:
    df["expected_answer"] = df["ground_truth"]

required_cols = ["question", "generated_answer", "expected_answer"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")


def _contexts_from_row(row):
    raw_ctx = row.get("retrieved_contexts_json", None)
    if isinstance(raw_ctx, str) and raw_ctx.strip() and raw_ctx.strip() != "nan":
        try:
            arr = json.loads(raw_ctx)
            if isinstance(arr, list):
                return [str(x) for x in arr if str(x).strip()]
        except Exception:
            pass

    val = str(row.get("retrieved_files_topk", ""))
    if val and val != "nan":
        return [x.strip() for x in val.split(";") if x.strip()]

    src = str(row.get("srs_file", ""))
    return [src] if src and src != "nan" else []

ragas_df = pd.DataFrame(
    {
        "question": df["question"].astype(str),
        "answer": df["generated_answer"].astype(str),
        "ground_truth": df["expected_answer"].astype(str),
        "contexts": df.apply(_contexts_from_row, axis=1),
        "srs_file": df["srs_file"].astype(str) if "srs_file" in df.columns else "unknown",
        "question_id": df["question_id"].astype(str) if "question_id" in df.columns else df.index.astype(str),
    }
)

ragas_dataset = Dataset.from_pandas(
    ragas_df[["question", "answer", "ground_truth", "contexts"]],
    preserve_index=False,
)

display(ragas_df.head(3))
print("Rows:", len(ragas_df))

ROOT: d:\Qbrainpython\QBrain\rag_lab
BENCHMARK_CSV: d:\Qbrainpython\QBrain\rag_lab\data\outputs\evaluation\rag_benchmark\benchmark_full_results.csv
OPENAI_API_KEY set: True


,question,answer,ground_truth,contexts,srs_file,question_id
0,What document is specified at the beginning of...,The document specified at the beginning of the...,The document is the ERTMS/ETCS Functional Requ...,[12.10.2005 \nOctober 2005 \n4.40 Oct \nUpdate...,2007 - ertms.pdf,q_ertms_01
1,What filing number is shown for this ERTMS/ETC...,The filing number shown for the ERTMS/ETCS FRS...,The filing number is ERA/ERTMS/003204.,[ERTMS/ETCS FRS v 5.00 \n \n \n \n \n \n \n \n...,2007 - ertms.pdf,q_ertms_02
2,What does the amendment history say about vers...,The amendment history indicates that version 5...,Version 5.00 is marked as the official release...,[12.10.2005 \nOctober 2005 \n4.40 Oct \nUpdate...,2007 - ertms.pdf,q_ertms_03


Rows: 45


In [7]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, answer_correctness
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import os
import warnings

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError("OPENAI_API_KEY is not set. Add it to rag_lab/.env then rerun setup cell.")

# ragas==0.4.x accepts these as already-initialized Metric objects.
# We silence deprecation warnings to keep notebook output clean.
warnings.filterwarnings("ignore", category=DeprecationWarning)
metrics = [faithfulness, answer_relevancy, answer_correctness]

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
emb = OpenAIEmbeddings(model="text-embedding-3-small")

result = evaluate(
    ragas_dataset,
    metrics=metrics,
    llm=llm,
    embeddings=emb,
    raise_exceptions=False,
)

result_df = result.to_pandas()
display(result_df.head())
print("Columns:", list(result_df.columns))

Evaluating:   0%|          | 0/135 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[5]: TimeoutError()
Exception raised in Job[14]: TimeoutError()
Exception raised in Job[11]: TimeoutError()
Exception raised in Job[2]: TimeoutError()
Exception raised in Job[17]: TimeoutError()
Exception raised in Job[20]: TimeoutError()
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,answer_correctness
0,What document is specified at the beginning of...,[12.10.2005 \nOctober 2005 \n4.40 Oct \nUpdate...,The document specified at the beginning of the...,The document is the ERTMS/ETCS Functional Requ...,1.000,0.877680,NaN
1,What filing number is shown for this ERTMS/ETC...,[ERTMS/ETCS FRS v 5.00 \n \n \n \n \n \n \n \n...,The filing number shown for the ERTMS/ETCS FRS...,The filing number is ERA/ERTMS/003204.,1.000,0.971480,NaN
2,What does the amendment history say about vers...,[12.10.2005 \nOctober 2005 \n4.40 Oct \nUpdate...,The amendment history indicates that version 5...,Version 5.00 is marked as the official release...,0.875,0.597858,0.579012
3,"According to the table of contents, which sect...",[ERTMS/ETCS FRS v 5.00 \n \n \n \n \nTABLE OF ...,"According to the table of contents, the sectio...","Section 2, ETCS Objectives, comes immediately ...",1.000,0.900425,NaN
4,Which subsection of General requirements addre...,[ERTMS/ETCS FRS v 5.00 \n \n11\n \n3.3 \nInten...,The subsection of General requirements that ad...,Subsection 3.7 addresses operation with existi...,1.000,0.923986,NaN


Columns: ['user_input', 'retrieved_contexts', 'response', 'reference', 'faithfulness', 'answer_relevancy', 'answer_correctness']


In [8]:
full = pd.concat([ragas_df.reset_index(drop=True), result_df.reset_index(drop=True)], axis=1)

metric_cols = [c for c in ["faithfulness", "answer_relevancy", "answer_correctness"] if c in full.columns]
if not metric_cols:
    raise ValueError("RAGAS metric columns not found in result dataframe.")

overall = full[metric_cols].mean().reset_index()
overall.columns = ["metric", "value"]

per_srs = full.groupby("srs_file", as_index=False)[metric_cols].mean()

full.to_csv(OUT_TABLES / "ragas_by_question.csv", index=False)
overall.to_csv(OUT_TABLES / "ragas_overall_summary.csv", index=False)
per_srs.to_csv(OUT_TABLES / "ragas_summary_by_srs.csv", index=False)

(OUT_TABLES / "ragas_overall_summary.tex").write_text(
    overall.to_latex(index=False, float_format=lambda x: f"{x:.4f}"),
    encoding="utf-8",
)
(OUT_TABLES / "ragas_summary_by_srs.tex").write_text(
    per_srs.to_latex(index=False, float_format=lambda x: f"{x:.4f}"),
    encoding="utf-8",
)

display(overall)
display(per_srs)
print("Saved RAGAS outputs to:", OUT_TABLES)

,metric,value
0,faithfulness,0.952186
1,answer_relevancy,0.807373
2,answer_correctness,0.605729


,srs_file,faithfulness,answer_relevancy,answer_correctness
0,2007 - ertms.pdf,0.954167,0.726199,0.578395
1,2008 - keepass.pdf,0.987500,0.806552,0.673685
2,2009 - inventory 2.0.pdf,0.924242,0.849012,0.616517
3,2010 - gparted.pdf,0.975000,0.901634,0.618012
4,JDECo_SRS.docx[1].pdf,0.931429,0.819452,0.562344


Saved RAGAS outputs to: d:\Qbrainpython\QBrain\rag_lab\results\tables
